In [6]:
import pandas as pd
import os
from tqdm import tqdm

target_sheet = "提出調査票43"
target_keyword = "死亡"

print("--- 2019〜2024年のデータを検索中 ---")
xlsx_files = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.xlsx'):
            xlsx_files.append(os.path.join(dirname, filename))

print(f"見つかったXLSXファイル数: {len(xlsx_files)} 個\n")

death_data_list = []

# 全ファイルをループ処理
for file_path in tqdm(xlsx_files):
    try:
        # Excelファイルの読み込み（列名は後で無視するのでそのまま読み込む）
        df = pd.read_excel(file_path, sheet_name=target_sheet, skiprows=5)
        
        # 1列目のデータから、空白を除去して「死亡」という文字と完全に一致する行を探す
        # (※NaNが含まれていてもエラーにならないよう文字型に変換)
        mask = df.iloc[:, 0].astype(str).str.strip() == target_keyword
        death_row = df[mask]
        
        if not death_row.empty:
            # 【重要】列名がどうなっていようと、必ず「左から2番目（インデックス1）」の値を合計値として取得する！
            total_deaths = death_row.iloc[0, 1]
            
            # ファイル名から病院名を抽出
            hospital_name = os.path.basename(file_path).split('_')[-1].replace('.xlsx', '')
            
            # 抽出した「病院名」と「合計値」だけを綺麗な辞書にしてリストに保存
            death_data_list.append({
                'hospital_name': hospital_name,
                '合計': total_deaths
            })
            
    except Exception as e:
        # シートがないなどのエラーはスキップ
        continue

# --- 集計とトップ10ランキングの表示 ---
if death_data_list:
    # 抽出したデータから新しい綺麗な表を作成
    master_df = pd.DataFrame(death_data_list)
    master_df_ = master_df # いったん逃がす
    
    # 文字列などが混ざっていても強制的に数値（エラーは0）に変換
    master_df['合計'] = pd.to_numeric(master_df['合計'], errors='coerce').fillna(0)
    
    # 病院名（hospital_name）ごとに死亡数の「合計」を足し合わせる
    ranking_df = master_df.groupby('hospital_name')['合計'].sum().reset_index()
    
    # 死亡数が多い順（降順）に並び替え
    ranking_df = ranking_df.sort_values(by='合計', ascending=False).reset_index(drop=True)
    
    print("\n\n🏆 死亡数トップ10病院 (複数年累計)")
    
    # 上位10件を綺麗に表示（インデックスを1〜10にする）
    top10_df = ranking_df.head(10)
    top10_df.index = range(1, 11)
    display(top10_df)
    
else:
    print("\nデータが抽出できませんでした。")

--- 2019〜2024年のデータを検索中 ---
見つかったXLSXファイル数: 502 個



100%|██████████| 502/502 [01:23<00:00,  6.01it/s]



🏆 死亡数トップ10病院 (複数年累計)


,hospital_name,合計
1,鶴川サナトリウム病院,36
2,医療法人社団純正会 青梅東部病院,31
3,医療法人社団小松会 聖パウロ病院,26
4,医療法人社団孝山会 滝山病院,23
5,稲城台病院,22
6,慈雲堂病院,20
7,こころのホスピタル町田,20
8,高月病院,18
9,根岸病院,18
10,平川病院,18


In [9]:
master_df_.head()

,hospital_name,合計
0,東京武蔵野病院,0
1,多摩平の森の病院,0
2,医療法人社団欣助会吉祥寺病院,0
3,国立研究開発法人国立国際医療研究センター病院,2
4,医療法人財団 富士病院,0
